In [ ]:
import spikeinterface.full as si
import matplotlib.pyplot as plt
import numpy as np
import probeinterface as pi
from pathlib import Path
import pandas as pd 
import os, sys
import shutil
from pprint import pprint 
import time as time
import json
%load_ext autoreload
%autoreload 2

import bombcell as bc

    
global_job_kwargs = dict(n_jobs=-3, chunk_duration="10s",progress_bar=True)
si.set_global_job_kwargs(**global_job_kwargs)


basefolder= r"H:\Data\raw\8186_naive"
base_path = Path(basefolder)



recording =  si.read_spikeglx(basefolder, stream_id='imec0.ap', load_sync_channel=False)
lfp = si.read_spikeglx(basefolder, stream_id='imec0.lf', load_sync_channel=False)
event =  si.read_spikeglx(basefolder, stream_id='nidq', load_sync_channel=False)
print(recording)

metapath = base_path / 'Meta'
if not os.path.isdir(metapath):
   os.makedirs(metapath)


In [ ]:
rec1 = si.bandpass_filter(recording)
rec1 = si.phase_shift(rec1)
bad_channel_ids, channel_labels = si.detect_bad_channels(rec1,method = 'coherence+psd')
print(bad_channel_ids)
rec1 = si.interpolate_bad_channels(recording=rec1, bad_channel_ids=bad_channel_ids)

rec1 = si.common_reference(rec1, operator="median", reference="global")
print(rec1)


%matplotlib widget
si.plot_traces({'raw':recording,'filtered':rec1}, backend='ipywidgets')

from spikeinterface.sorters import installed_sorters
installed_sorters()
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
torch.cuda.get_device_name(0)

In [ ]:
Sorting_KS4 = si.run_sorter(sorter_name="kilosort4", recording=rec1, folder=basefolder + str('/sorted'),remove_existing_folder=True)
analyzer = si.create_sorting_analyzer(Sorting_KS4, rec1, sparse=True, format='binary_folder',folder=basefolder +str('/analyzer'))

analyzer.compute(['random_spikes', 'waveforms', 'templates', 'noise_levels','unit_locations','correlograms'],**global_job_kwargs)
analyzer.compute('spike_amplitudes')
analyzer.compute('principal_components', n_components = 5, mode="by_channel_local",**global_job_kwargs)
analyzer.compute(["spike_locations", "template_metrics", "quality_metrics"],**global_job_kwargs)

plot_path =base_path/'plots'
spike_times_path = base_path / 'spike_times'


# Load Data

sf = recording.sampling_frequency

# 1. Bombcell & Labels
bc_results = si.bombcell_label_units(analyzer, thresholds=si.bombcell_get_default_thresholds())
labels = bc_results["bombcell_label"].values
analyzer.sorting.set_property("bombcell_label", labels)

# Save Labels Plot
w = si.plot_unit_labels(analyzer, labels, ylims=(-300, 100))
w.figure.savefig(plot_path / "bombcell_labels.png")
plt.close(w.figure)



spikes = pd.DataFrame(analyzer.sorting.to_spike_vector())
label_map = pd.Series(labels, index=range(len(analyzer.unit_ids)))
spikes['label'] = spikes['unit_index'].map(label_map)
spikes['time_s'] = spikes['sample_index'] / analyzer.sampling_frequency
spike_df = spikes.drop(columns=['segment_index'], errors='ignore')
spike_df.to_csv(spike_times_path / 'spike_times.csv', index=False)


# Convert to records and save
structured_array = spike_df.to_records(index=False)


np.save(spike_times_path / 'spike_times.npy', structured_array)



    
    # Convert to records and save
    structured_array = spike_df.to_records(index=False)
    
    
    np.save(spike_times_path / 'spike_times.npy', structured_array)

In [ ]:

from scipy.signal import medfilt
from scipy.interpolate import interp1d


def ttl_from_analog(signal, fs, hysteresis=0.1, filt_kernel=5):
    signal_f = medfilt(signal, kernel_size=filt_kernel)
    v_low = np.percentile(signal_f, 5)
    v_high = np.percentile(signal_f, 95)
    v_lo = v_low + hysteresis * (v_high - v_low)
    v_hi = v_high - hysteresis * (v_high - v_low)

    digital = np.zeros_like(signal_f, dtype=np.int8)
    state = 0
    for i, v in enumerate(signal_f):
        if state == 0 and v > v_hi: state = 1
        elif state == 1 and v < v_lo: state = 0
        digital[i] = state
    d = np.diff(digital)
    rising_idx = np.where(d == 1)[0] + 1
    falling_idx = np.where(d == -1)[0] + 1
    return digital, rising_idx, falling_idx

def quadrature_speed_direction(sigA, sigB, fs, pulses_per_rev=360, hysteresis=0.1):
    sig_len = len(sigA)
    time_full = np.arange(sig_len) / fs
    digA, rising_A, _ = ttl_from_analog(sigA, fs, hysteresis)
    digB, _, _ = ttl_from_analog(sigB, fs, hysteresis)
    if len(rising_A) < 2:
        return time_full * 1000, np.zeros(sig_len), np.zeros(sig_len)
    t_edges = rising_A / fs
    direction_edges = np.where(digB[rising_A] == 0, 1, -1)
    deg_per_edge = 360.0 / pulses_per_rev
    dtheta = direction_edges * deg_per_edge
    dt = np.diff(t_edges)
    speed_edges = np.diff(np.cumsum(dtheta)) / dt
    dir_edges = direction_edges[1:]
    t_mid = (t_edges[1:] + t_edges[:-1]) / 2
    f_speed = interp1d(t_mid, speed_edges, kind='previous', bounds_error=False, fill_value=0)
    f_dir = interp1d(t_mid, dir_edges, kind='previous', bounds_error=False, fill_value=0)
    return time_full * 1000, f_speed(time_full), f_dir(time_full)

def extract_ttl_from_bit(digital_word, bit, sampling_rate, plot_path, savename, plot=True):
    ttl_signal = (digital_word >> bit) & 1
    time_axis = np.arange(len(ttl_signal)) / sampling_rate
    if plot:
        plt.figure(figsize=(15, 3))
        plt.plot(time_axis, ttl_signal)
        plt.title(f'Isolated Bit {bit} ({savename})')
        plt.savefig(plot_path / f"ttl_bit_{bit}_{savename}.png")
        plt.close()
    diff = np.diff(ttl_signal)
    rising_idx = np.where(diff > 0)[0]
    falling_idx = np.where(diff < 0)[0]
    edges = np.concatenate([rising_idx / sampling_rate, falling_idx / sampling_rate])
    types = (['rising'] * len(rising_idx)) + (['falling'] * len(falling_idx))
    return pd.DataFrame({'timestamps': edges, 'edge_type': types}).sort_values('timestamps').reset_index(drop=True)

def extract_and_save_ttl_events(data, bit_name_pairs, save_path, plot_path):
    # CRITICAL: return_scaled=False so bitwise >> works on integers
    digital_signals = data.get_traces(return_in_uV=False)
    digital_word = digital_signals[:, 8]
    sampling_rate = data.get_sampling_frequency()
    for bit, savename in bit_name_pairs:
        ttl_df = extract_ttl_from_bit(digital_word, bit, sampling_rate, plot_path, savename)
        ttl_df.to_csv(save_path / f"{savename}.csv", index=False)
        
 # 4. Camera & Audio/State TTLs
cam_signal = event.get_traces(channel_ids=[event.get_channel_ids()[3]]).squeeze()
_, _, falling_A = ttl_from_analog(cam_signal, sf)
if len(falling_A) > 0:
    pd.DataFrame({'camttl': [falling_A[0] / sf]}).to_csv(metapath / 'camttl.csv', index=False)

pairs = [(1, 'State_changes'), (3, 'Audio')]
extract_and_save_ttl_events(event, pairs, metapath, plot_path)